# Data Cleaning Assignment

### Q1. Missing Data Identification
### Scenario:
##### The hospital suspects incomplete patient records.
### Task:
- Identify missing values in each column
- Calculate percentage of missing data

### Answer:
##### Steps:
1. Count missing values in each column using `isnull().sum()`
2. Calculate percentage of missing data for each column

In [1]:
import pandas as pd

In [2]:
# Loading dataset
df = pd.read_csv('healthcare_data_cleaning_dataset.csv')

In [3]:
# Step 1 : Counting missing values in each column
missing_values = df.isnull().sum()
print('Missing Values :', missing_values)

Missing Values : Patient_ID              0
Age                   600
Gender                  0
City                    0
Diagnosis               0
Hospital_Visits         0
Treatment_Cost        593
Insurance_Coverage      0
Admission_Date          0
dtype: int64


In [4]:
# 2. Calculate percentage of missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
print('Percentage  of missing values:', missing_percentage)

Percentage  of missing values: Patient_ID             0.000000
Age                   11.764706
Gender                 0.000000
City                   0.000000
Diagnosis              0.000000
Hospital_Visits        0.000000
Treatment_Cost        11.627451
Insurance_Coverage     0.000000
Admission_Date         0.000000
dtype: float64


### Q2. Handling Missing Age
### Scenario:
#### Age is critical for medical analysis, but some values are missing.
### Task:
- Replace missing Age values with an appropriate method
- Justify your choice (mean/median)

#### Answer: 
I am replaceing missing Age values with the **median age** of the dataset, **Reason** is Since Age is a numerical variable and can contain outliers (very young or very old patients), using the **median** is a better choice than the mean.


In [5]:
# Replaceing missing Age values with the median age of the dataset
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)

In [6]:
# Verifying handled missing values
df['Age'].isnull().sum()

np.int64(0)

#### Justification :
- Mean can be affected by extreme values (outliers)
- Median represents the middle value and is more robust

### Q3. Handling Missing Treatment Cost
### Scenario:
#### Treatment cost is highly skewed due to expensive treatments.
### Task:
- Handle missing Treatment_Cost values.
- Choose the correct imputation method and explain why.

#### Answer :
Since treatment costs are typically **highly skewed** (a few very expensive treatments and many lower-cost ones), using the **median** is the most appropriate method.

In [7]:
# Calculate median treatment cost
median_cost = df['Treatment_Cost'].median()

In [8]:
# Fill missing values with median
df['Treatment_Cost'] = df['Treatment_Cost'].fillna(median_cost)

In [9]:
# Verify missing values are handled
df['Treatment_Cost'].isnull().sum()

np.int64(0)

#### Reason:
- Mean is sensitive to extreme high values (outliers)
- Median is robust and better represents the central tendency in skewed data

Therefore, missing values in Treatment_Cost are replaced with the **median cost**.

### Q4. Duplicate Patient Records
### Scenario:
#### Some patient records were entered multiple times.
### Task:
- Identify duplicate rows
- Remove duplicates
- Compare dataset size before and after

#### Answer : 
Step wise tprocess followed to ensures data accuracy and prevents biased analysis due to repeated entries is as follows :

In [11]:
# No. of rows with duplicates.
before_rows = df.shape[0]
print("Rows before removing duplicates:", before_rows)

Rows before removing duplicates: 5100


In [12]:
# No. of duplicate rows
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 99


In [13]:
# Removeing duplicate rows
df = df.drop_duplicates()

In [14]:
# Dataset size after removing duplicates
after_rows = df.shape[0]
print("Rows after removing duplicates:", after_rows)

Rows after removing duplicates: 5001


In [16]:
# Verify duplicates are removed
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


### Q5. Invalid Age Values (Data Quality Check)
### Scenario:
#### Some patients have unrealistic age values (e.g., >100 or <0).
### Task:
- Detect such records
- Decide whether to remove or correct them

#### Answer:
Since these values are clearly incorrect and cannot be reliably corrected, it is safer to remove them rather impute or guess values. TO check for unrealistic values in the Age column we will consider the following conditions:
##### Invalid conditions -
- Age < 0 (not possible)
- Age > 100 (rare and likely data entry error)

In [18]:
# Step 1 : Detecting invalid age rows in dataset
invalid_age = df[(df['Age'] < 0) | (df['Age'] > 100)]
print("Number of invalid age records:", invalid_age.shape)

Number of invalid age records: (0, 9)


#### Interpretation:
There are **no unrealistic age values** (i.e., no values < 0 or > 100) in the dataset.
The result `(0, 9)` means:
- 0 rows → No invalid age records found
- 9 columns → Total columns in the dataset

#### Conclusion:
- No action is required for this step
- The Age column already satisfies data quality conditions

### Q6. Outlier Detection (Treatment Cost)
### Scenario:
#### Extreme treatment costs are affecting analysis.
### Task:
- Detect outliers using IQR method
- Display number of outliers

#### Answer :
Steps:
1. Calculate Q1 (25th percentile) and Q3 (75th percentile)
2. Compute IQR = Q3 - Q1
3. Define bounds:
   - Lower Bound = Q1 - 1.5 * IQR
   - Upper Bound = Q3 + 1.5 * IQR
4. Detect number of values outside these bounds as outliers and print the value.

In [19]:
# Step 1 :
Q1 = df['Treatment_Cost'].quantile(0.25)
Q3 = df['Treatment_Cost'].quantile(0.75)

In [20]:
# Step 2 :
IQR = Q3 - Q1

In [21]:
# Step 3 :
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [22]:
# Step 4 :
outliers = df[(df['Treatment_Cost'] < lower_bound) |
              (df['Treatment_Cost'] > upper_bound)]
print("Number of outliers:", outliers.shape[0])

Number of outliers: 50


### Q7. Outlier Treatment
### Scenario:
#### The business team wants to retain all records.
### Task:
- Apply capping (Winsorization) on Treatment_Cost
- Use 5th and 95th percentile

#### Answer :
Since the business wants to retain all records, we do not remove outliers. Instead, we limit extreme values.
Steps:
1. Calculate 5th percentile (lower limit) and Calculate 95th percentile (upper limit) 
2. Replace values:
   - Below 5th percentile → set to 5th percentile
   - Above 95th percentile → set to 95th percentile
3. Verify results
This approach reduces the impact of extreme values while preserving all data points.

In [23]:
# Step 1 :
lower_cap = df['Treatment_Cost'].quantile(0.05)
upper_cap = df['Treatment_Cost'].quantile(0.95)

In [24]:
# Step 2 :
df['Treatment_Cost'] = df['Treatment_Cost'].clip(lower=lower_cap, upper=upper_cap)

In [25]:
# Step 3 :
print("New min value:", df['Treatment_Cost'].min())
print("New max value:", df['Treatment_Cost'].max())

New min value: 3238.0
New max value: 47948.0


### Q8. Transformation
### Scenario:
#### Treatment cost is highly skewed.
### Task:
- Apply log transformation
- Create a new column
- Compare before vs after distribution

#### Answer :
Steps:
1. Apply log transformation using `log1p()` and Create a new column `Log_Treatment_Cost`
2. Compare distribution before and after log transformation.

In [28]:
# Step 1 :
import numpy as np
df['Log_Treatment_Cost'] = np.log1p(df['Treatment_Cost'])

In [34]:
# Step 2:

print("Comparesion of distribution before and after Log Transformation :")
comparison = pd.concat([
    df['Treatment_Cost'].describe(), 
    df['Log_Treatment_Cost'].describe()
], axis=1)

print(comparison)

Comparesion of distribution before and after Log Transformation :
       Treatment_Cost  Log_Treatment_Cost
count     5001.000000         5001.000000
mean     25218.878624            9.918335
std      13580.445490            0.751774
min       3238.000000            8.083020
25%      13766.000000            9.530030
50%      24797.000000           10.118518
75%      36542.000000           10.506245
max      47948.000000           10.777893


### Q9. Time-Based Missing Handling
### Scenario:
#### Admission dates should follow a logical sequence.
### Task:
- Sort data by Admission_Date
- Apply forward fill or backward fill where appropriate
- Justify your choice

#### Answer :
##### Steps -
1. Converted `Admission_Date` to datetime format for proper sorting
2. Checked missing values using `isnull().sum()` (Result shows **no missing values in any column**)
3. Sorted dataset by `Admission_Date` to maintain chronological order
4. Check if Admission_Date is sorted properly

In [35]:
# Step 1 :
df['Admission_Date'] = pd.to_datetime(df['Admission_Date'])

In [36]:
# Step 2 :
df.isnull().sum()

Patient_ID            0
Age                   0
Gender                0
City                  0
Diagnosis             0
Hospital_Visits       0
Treatment_Cost        0
Insurance_Coverage    0
Admission_Date        0
Log_Treatment_Cost    0
dtype: int64

In [37]:
# Step 3 :
df = df.sort_values(by='Admission_Date')

In [40]:
# Step 4 :
print("Dates after sorting:")
print(df['Admission_Date'])

Dates after sorting:
2932   2023-01-01
4617   2023-01-01
2642   2023-01-01
4837   2023-01-01
2702   2023-01-01
          ...    
1942   2023-12-31
1723   2023-12-31
2593   2023-12-31
3013   2023-12-31
3133   2023-12-31
Name: Admission_Date, Length: 5001, dtype: datetime64[ns]


#### Justification:
- Since there are **no missing values**, forward fill or backward fill is **not required**
- Applying fill methods when no missing data exists is unnecessary
- Only sorting is sufficient to maintain logical time sequence